In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [3]:
tf.random.set_seed(42)

In [4]:
np.random.seed(42)

In [5]:
VOCAB_SIZE = 10000

In [6]:
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

In [7]:
print(f"Training samples: {len(X_train_raw)}, Testing samples: {len(X_test_raw)}")

Training samples: 25000, Testing samples: 25000


In [8]:
print(f"Example label values: {np.unique(y_train)} (0=negative, 1=positive)")

Example label values: [0 1] (0=negative, 1=positive)


In [9]:
np.unique(y_train)

array([0, 1])

In [10]:
review_lengths = [len(x) for x in X_train_raw]

In [11]:
print(f"   Shortest review: {min(review_lengths)} words")

   Shortest review: 11 words


In [12]:
print(f"   Longest review: {max(review_lengths)} words")

   Longest review: 2494 words


In [13]:
print(f"   Average review: {np.mean(review_lengths):.2f} words")

   Average review: 238.71 words


In [14]:
print(f"  Example raw (encoded) review, first 10 tokens: {X_train_raw[0][:10]}")

  Example raw (encoded) review, first 10 tokens: [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]


In [15]:
plt.figure(figsize=(8, 4))
plt.hist(review_lengths, bins=50, color="steelblue")
plt.axvline(200, color="red", linestyle="--", label="cutoff we'll use (200 words)")
plt.title("Distribution of review lengths (in words)")
plt.xlabel("Number of words")
plt.ylabel("Number of reviews")
plt.legend()
plt.tight_layout()
plt.savefig("review_length_distribution.png")
plt.close()
print("  Saved plot: review_length_distribution.png")

  Saved plot: review_length_distribution.png


In [16]:
MAX_LEN = 200

X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding="pre", truncating="pre")
X_test = pad_sequences(X_test_raw, maxlen=MAX_LEN, padding="pre", truncating="pre")

print("\nSTEP 3: Preprocessing done")
print(f"  X_train shape: {X_train.shape}  (num_reviews, fixed_length)")
print(f"  X_test shape:  {X_test.shape}")


STEP 3: Preprocessing done
  X_train shape: (25000, 200)  (num_reviews, fixed_length)
  X_test shape:  (25000, 200)


In [17]:
print(f"Training samples: {len(X_train_raw)}, Testing samples: {len(X_test_raw)}")

Training samples: 25000, Testing samples: 25000


In [18]:
X_train.shape, X_test.shape

((25000, 200), (25000, 200))

In [19]:
EMBED_DIM = 32
LSTM_UNITS = 64

In [20]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [21]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [23]:
EPOCHS = 5
BATCH_SIZE = 128

lstm = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=2
)

Epoch 1/5
157/157 - 41s - 262ms/step - accuracy: 0.7099 - loss: 0.5605 - val_accuracy: 0.7836 - val_loss: 0.4768
Epoch 2/5
157/157 - 29s - 188ms/step - accuracy: 0.8294 - loss: 0.3981 - val_accuracy: 0.8342 - val_loss: 0.3885
Epoch 3/5
157/157 - 29s - 184ms/step - accuracy: 0.8723 - loss: 0.3148 - val_accuracy: 0.8318 - val_loss: 0.3876
Epoch 4/5
157/157 - 29s - 184ms/step - accuracy: 0.8902 - loss: 0.2845 - val_accuracy: 0.8336 - val_loss: 0.3874
Epoch 5/5
157/157 - 29s - 186ms/step - accuracy: 0.9054 - loss: 0.2508 - val_accuracy: 0.8262 - val_loss: 0.4083


In [24]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_acc}")

Test Loss: 0.40240371227264404, Test Accuracy: 0.8295199871063232


In [25]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(lstm.history['accuracy'], label='Train Accuracy')
axes[0].plot(lstm.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(lstm.history['loss'], label='Train Loss')
axes[1].plot(lstm.history['val_loss'], label='Validation Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig("lstm_training_history.png")
plt.close()

In [26]:
word_index = imdb.get_word_index()

reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review if i != 0) 

sample_idx = 7
sample = X_test[sample_idx].reshape(1, -1)
predicted_prob = model.predict(sample, verbose=0)[0][0]
predicted_label = 'POSITIVE' if predicted_prob > 0.5 else 'NEGATIVE'
true_label = 'POSITIVE' if y_test[sample_idx] == 1 else 'NEGATIVE'


print("   Review text")
print(" ", decode_review(X_test_raw[sample_idx][-60:]), "...")
print(f"   True label: {true_label}")
print(f"   Predicted label: {predicted_label}")

   Review text
  its a terrible storyline there are 3 main musical pieces all of which are rubbish bad songs and dreadful choreography its just an extremely boring film bing has too many words in each sentence and delivers them in an almost irritating manner its not funny ever but its meant to be bing and joan have done much better than this ...
   True label: NEGATIVE
   Predicted label: NEGATIVE
